# routes/selection

> Route handlers for selection queue management (remove, reorder, clear)

In [ ]:
#| default_exp routes.selection

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict, Tuple, Callable
from pathlib import Path

from fasthtml.common import APIRouter

from cjm_fasthtml_file_browser.core.config import FileBrowserConfig
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcription_source_select.models import SourceSelectUrls
from cjm_transcription_source_select.routes.core import (
    get_step_state, update_step_state, get_session_id_from_sess
)
from cjm_transcription_source_select.components.selection_panel import render_selection_panel
from cjm_transcription_source_select.components.file_browser_panel import (
    get_browser_state, sync_browser_selection, render_browser_panel
)

## OOB Browser Panel Helper

When selection changes via remove/clear, the browser panel needs to update its selection highlights.

In [ ]:
#| export
def _render_oob_browser(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    session_id: str,  # Session identifier
    selected_files: list,  # Current selected files
):  # Browser panel with OOB attribute set
    """Render browser panel as OOB swap to update selection highlights."""
    step_state = get_step_state(state_store, workflow_id, session_id)
    browser_state = get_browser_state(step_state, home_path)
    sync_browser_selection(browser_state, selected_files)

    # Save updated browser state
    update_step_state(
        state_store, workflow_id, session_id,
        browser_state=browser_state.to_dict(),
    )

    browser = render_browser_panel(
        browser_state=browser_state,
        config=config,
        provider=provider,
        navigate_url=urls.navigate,
        select_url=urls.select,
        home_path=home_path,
    )
    browser.attrs["hx-swap-oob"] = "outerHTML"
    return browser

## Remove Handler

In [ ]:
#| export
def _handle_remove(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
    path: str,  # File path to remove
):  # (selection panel, OOB browser panel)
    """Remove a file from the selection."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    selected_files = step_state.get("selected_files", [])

    selected_files = [f for f in selected_files if f["path"] != path]

    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=selected_files,
        verified=False,
        committed_audio_paths=[],
    )

    selection = render_selection_panel(selected_files, urls)
    browser_oob = _render_oob_browser(
        state_store, provider, config, workflow_id, home_path, urls, session_id, selected_files
    )
    return selection, browser_oob

## Reorder Handler

## Clear Handler

In [ ]:
#| export
async def _handle_reorder(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    request,  # FastHTML request
    sess,  # FastHTML session
):  # Rendered selection panel
    """Reorder selected files based on SortableJS drag result."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    selected_files = step_state.get("selected_files", [])

    # Extract new order from form data (Hidden inputs sent in DOM order)
    form_data = await request.form()
    new_order_paths = form_data.getlist("item")

    # Rebuild list in new order
    path_to_file = {f["path"]: f for f in selected_files}
    reordered = [path_to_file[p] for p in new_order_paths if p in path_to_file]

    # Add any files not in the form data (shouldn't happen, but safe)
    reordered_paths = {f["path"] for f in reordered}
    for f in selected_files:
        if f["path"] not in reordered_paths:
            reordered.append(f)

    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=reordered,
    )

    return render_selection_panel(reordered, urls)

In [ ]:
#| export
def _handle_clear(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
):  # (selection panel, OOB browser panel)
    """Clear all selected files."""
    session_id = get_session_id_from_sess(sess)

    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=[],
        verified=False,
        committed_audio_paths=[],
    )

    selection = render_selection_panel([], urls)
    browser_oob = _render_oob_browser(
        state_store, provider, config, workflow_id, home_path, urls, session_id, []
    )
    return selection, browser_oob

## Router Initialization

In [ ]:
#| export
def init_selection_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider (for OOB browser updates)
    config: FileBrowserConfig,  # Browser configuration (for OOB browser updates)
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle
    home_path: str = "",  # Home directory path
    prefix: str = "/selection",  # Route prefix
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize selection queue management routes."""
    router = APIRouter(prefix=prefix)
    _home = home_path or str(Path.home())

    @router
    def remove(sess, path: str = ""):
        """Remove a file from the selection."""
        if not path:
            return render_selection_panel(
                get_step_state(state_store, workflow_id, get_session_id_from_sess(sess)).get("selected_files", []),
                urls
            )
        return _handle_remove(state_store, provider, config, workflow_id, _home, urls, sess, path)

    @router
    async def reorder(request, sess):
        """Reorder selected files via SortableJS."""
        return await _handle_reorder(state_store, workflow_id, urls, request, sess)

    @router
    def clear(sess):
        """Clear all selected files."""
        return _handle_clear(state_store, provider, config, workflow_id, _home, urls, sess)

    routes = {
        "remove": remove,
        "reorder": reorder,
        "clear": clear,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()